In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType

In [0]:
df = spark.read.option("header", "true").option("inferSchema", "true").option("quote", '"').option("escape", '"').csv("abfss://bronze@deassessment.dfs.core.windows.net/loan_list")

# Verify the schema and data
df.printSchema()
df.display()

In [0]:
# 1. Define the schema matching your JSON fields and data types
borrower_schema = StructType([
    StructField("credit_score", IntegerType(), True),
    StructField("employment", StringType(), True),
    StructField("annual_income", IntegerType(), True),
    StructField("years_employed", IntegerType(), True)
])

# 2. Parse the JSON string column into a Struct
df_parsed = df.withColumn("borrower_info2", F.from_json(F.col("borrower_info"), borrower_schema))

df_parsed.filter(F.col("borrower_info2.annual_income").isNull()).display()

In [0]:
# 1. First, fix the raw string anomalies step-by-step
df_cleaned = df.withColumn(
    "borrower_info_clean",
    # Fix 1: Replace "=" with ":" for credit_score
    F.regexp_replace(F.col("borrower_info"), r'"credit_score"\s*=\s*', '"credit_score": ')
).withColumn(
    "borrower_info_clean",
    # Fix 2: Remove trailing commas before a closing brace (e.g., ,} -> })
    F.regexp_replace(F.col("borrower_info_clean"), r',\s*}', '}')
).withColumn(
    "borrower_info_clean",
    # Fix 3: If the row doesn't end with a closing brace '}', append it
    F.when(
        F.col("borrower_info_clean").endswith("}"),
        F.col("borrower_info_clean")
    ).otherwise(
        F.concat(F.col("borrower_info_clean"), F.lit("}"))
    )
)

# 2. Define your schema
borrower_schema = StructType([
    StructField("credit_score", IntegerType(), True),
    StructField("employment", StringType(), True),
    StructField("annual_income", LongType(), True),
    StructField("years_employed", IntegerType(), True)
])

# 3. Parse and flatten the corrected strings
df_parsed = df_cleaned.withColumn("borrower_struct", F.from_json(F.col("borrower_info_clean"), borrower_schema))

df_parsed.display()

In [0]:
df_parsed.filter(F.col("borrower_struct.annual_income").isNull()).display()

In [0]:
df_parsed=df_parsed.drop("borrower_info").drop("borrower_info_clean").withColumnRenamed("borrower_struct", "borrower_info")
df_parsed.display()

In [0]:
df_parsed.select("loan_id").distinct().count()


In [0]:
df_parsed.count()

In [0]:
df_parsed.groupBy("loan_id").count().orderBy(F.col("count").desc()).display()

In [0]:
df_parsed.filter(F.col("loan_id")=='L0001482').display()

based above data,data still unclean because loan_id should be unique (count =1 )

In [0]:
df_parsed.display()

In [0]:
import pyspark.sql.functions as F

# Define the regex pattern: 
# ^ = start of string, [A-Za-z] = one alphabet, \d{6} = exactly 6 digits, $ = end of string
regex_pattern = r"^[L]\d{7}$"

# Filter for rows that DO NOT match the pattern
invalid_customers_df = df_parsed.filter(~F.col("loan_id").rlike(regex_pattern))

invalid_customers_df.display()

In [0]:
import pyspark.sql.functions as F

# Define the regex pattern: 
# ^ = start of string, [A-Za-z] = one alphabet, \d{6} = exactly 6 digits, $ = end of string
regex_pattern = r"^[C]\d{6}$"

# Filter for rows that DO NOT match the pattern
invalid_customers_df = df_parsed.filter(~F.col("customer_id").rlike(regex_pattern))

invalid_customers_df.display()

In [0]:
import pyspark.sql.functions as F

# Transforms "a12345b" to "A12345b"
df_parsed = df_parsed.withColumn("product_type", F.initcap(F.col("product_type")))

In [0]:
df_parsed.select("product_type").distinct().display()

In [0]:

# [$,] means: match either '$' or ',' 
# We replace them with an empty string "" to delete them
df_parsed = df_parsed.withColumn("principal_amount", F.regexp_replace(F.col("principal_amount"), r"[\$,]", ""))

df_parsed.display()

In [0]:
df_parsed.printSchema()

In [0]:
import pyspark.sql.functions as F

# Overwrite the column by casting it to "double"
df_parsed = df_parsed.withColumn("principal_amount", F.col("principal_amount").cast("double"))

df_parsed.printSchema()

In [0]:
# 1. Define all the unique date formats currently hidden in your data
# Order matters: Put your most common or strict formats first
date_formats = [
    "yyyy-MM-dd",        # e.g., 2026-06-24
    "dd/MM/yyyy",        # e.g., 24/06/2026
    "MM/dd/yyyy",        # e.g., 06/24/2026
    "yyyy/MM/dd",        # e.g., 2026/06/24
    "dd-MMM-yyyy",       # e.g., 24-Jun-2026
    "yyyyMMdd"           # e.g., 20260624
]

# 2. Build a list of to_date() transformations dynamically
parse_attempts = [F.try_to_date(F.col("origination_date"), fmt) for fmt in date_formats]

# 3. Coalesce them together. The first format that successfully parses wins!
df_parsed = df_parsed.withColumn("origination_date", F.coalesce(*parse_attempts))

df_parsed.display()
df_parsed.printSchema()

In [0]:
df_parsed.select("origination_channel").distinct().display()

In [0]:
df_parsed.select("status").distinct().display()

In [0]:
df_parsed=df_parsed.distinct()

In [0]:
df_parsed.select("loan_id").distinct().count()

In [0]:
df_parsed.count()

In [0]:
# 1. Define the window specification
window_spec = Window.partitionBy("customer_id").orderBy(F.col("interest_rate").asc())

# 2. Add a row number and filter for the first row
df_lowest_rate = df_parsed.withColumn("row_num", F.row_number().over(window_spec)) \
                   .filter(F.col("row_num") == 1) \
                   .drop("row_num")

df_lowest_rate.display()

In [0]:
df_lowest_rate.select("customer_id").distinct().count()

In [0]:
df_lowest_rate.count()

In [0]:
df_lowest_rate.write.format("delta").mode("overwrite").save("abfss://silver@deassessment.dfs.core.windows.net/loans/")

In [0]:
df_parsed.display()